In [ ]:
# Änderungen:
# Komplettes Feature Engineering in eigene Funktion ausgelagert: aufrufen, dann split
# Zusammenlegen einiger Codeblöcke
# Permutation Importances Ausgabe
# Bei Modellen: Zusammenführung wiederholendem Code als Funktion:
#   Hier dann auch Funktion aus Aufgabe 5 für Kosten / Schwellwert aufrufen
#   Speicherung der Modelle in dict:
#       Auswahl Modell nach besten y_valid acc, auf welchem dann kompletter Datensatz trainiert wird
#       Auswahl geringste Kosten / Schwellwert


# Hotels getrennt trainieren
# Gurobi Lösung eingebaut, klappt aber noch nicht
# Ausgabe aller Modelle als df


# --------------
# Ideen für Features

# Monate / Saison als Feature über anreisedatum
# Frühere Buchung / Stornierung in Prozent
# Frühere Stornierung in Quadrat
# Vorlaufzeit = 0 als Feature (vor Ort)
# Aufenthaltsdauer
# Erwachsene / Kinder / Babys OHE

# (1) Betrachtung der Daten

In [ ]:
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

df_train = pd.read_csv("data/buchungen.csv")
df_test = pd.read_csv("data/buchungen_test.csv")  # Test-Datensatz ohne Storniert-Status

display(df_train.dtypes)  # Ausgabe der Spalten und deren jeweiligen Datentyp
display(df_train)

In [ ]:
df = df_train.copy()

cancel_rate = float(df.loc[df["storniert"] == 1]["storniert"].count() / df["storniert"].count() * 100)

df_type = df[df["hotel_typ"] == "Ferienhotel"]
cancel_rate_holiday = float(df_type[df["storniert"] == 1]["storniert"].count() / df_type["storniert"].count() * 100)

df_type = df[df["hotel_typ"] == "Stadthotel"]
cancel_rate_city = float(df_type[df["storniert"] == 1]["storniert"].count() / df_type["storniert"].count() * 100)

print(
    f"Stornierungsraten:\n"
    f"Gesamt:       {cancel_rate:.2f} %\n"
    f"Ferienhotel:  {cancel_rate_holiday:.2f} %\n"
    f"Stadthotel:   {cancel_rate_city:.2f} %\n"
)

# TODO: Auswahl von Spalten zum plotten
# for column in df.select_dtypes(include=np.number).columns.drop(["storniert", "stammgast", "agentur_id"]):
for column in ["vorlaufzeit_tage", "warteliste_tage", "durchschn_tagespreis"]:
    _, axes = plt.subplots(1, 2, figsize=(14, 6))

    axes[0].hist(df[column].dropna(), bins="auto", color="orange", edgecolor="black")
    axes[0].set_ylabel("Häufigkeit")

    axes[1].boxplot(df[column].dropna(), vert=False)

    plt.title(f"Verteilung von {column}")
    plt.tight_layout()
    plt.show()

# (2 / 3) Vorverarbeitung und Feature Engineering

In [ ]:
# Anzahl Feiertage im Buchungszeitraum
df_holidays = pd.read_csv("data/portugal_feiertage.csv")
df_holidays["datum"] = pd.to_datetime(df_holidays["datum"])
df_holidays = df_holidays.set_index("datum")


def map_holidays(df: pd.DataFrame) -> None:
    """Berechnet die Anzahl an Feiertage innerhalb eines Buchungszeitraumes als neues Feature."""
    df = df.copy()
    df["abreisedatum"] = df["anreisedatum"] + pd.to_timedelta((df["naechte_wochenende"] + df["naechte_wochentag"]), unit="D")

    df["anzahl_feiertage"] = 0
    for d in df_holidays.index:
        df["anzahl_feiertage"] += ((df["anreisedatum"] <= d) & (df["abreisedatum"] >= d)).astype(int)

    return df.drop(columns=["abreisedatum"])

In [ ]:
import country_converter as coco

# Manche Einträge kann country-converter nicht zuordnen
# Die Tabelle: https://de.wikipedia.org/wiki/ISO-3166-1-Kodierliste
# hat eine vollständige Liste und führt auch veraltete Codes auf
MANUAL_CONTINENT_MAPPING = {
    "TMP": "Asia",  # Osttimor (veraltet), aktueller Code: TLS
}


def add_country_features(df: pd.DataFrame) -> pd.DataFrame:
    """Ersetzt "herkunftsland" durch zwei neue Features.

    "herkunftskontinent": Kontinent des Herkunftslandes.
    "aus_portugal":       1 wenn Herkunftsland Portugal (PRT), sonst 0.
    """
    cc = coco.CountryConverter()
    df = df.copy()

    # Binäres Feature: Gast kommt aus Portugal
    # TODO: Auswahl weiterer Länder
    df["aus_portugal"] = (df["herkunftsland"] == "PRT").astype(bool)

    def map_continent(code):
        if pd.isna(code):
            return "Unbekannt"
        if code in MANUAL_CONTINENT_MAPPING:
            return MANUAL_CONTINENT_MAPPING[code]
        return cc.convert(code, to="continent")

    df["herkunftskontinent"] = df["herkunftsland"].apply(map_continent)
    return df.drop(columns=["herkunftsland"])

In [ ]:
def features(df: pd.DataFrame, drop: list | None = None) -> pd.DataFrame:
    """Generiert oder transformiert neue Features."""
    drop = [] if drop is None else drop
    df = df.copy()

    df["anreisedatum"] = pd.to_datetime(df["anreisedatum"])
    df["stammgast"] = df["stammgast"].astype(str)  # Stammgast ist kategorisch
    df["agentur_id"] = df["agentur_id"].astype(str)  # Agenturen sind kategorisch

    df = map_holidays(df)
    df = add_country_features(df)

    return df.drop(columns=[*drop, "anreisedatum"])

In [ ]:
from sklearn.model_selection import train_test_split

# Duplizierte Datenpunkte entfernen (hier vor dem Split)
# TODO: Fragen, ob Duplikate unsaubere Daten sind oder tatsächlich Bedeutung haben: 2 verschiedene Gäste haben die gleiche Buchung
#           Wurde geklärt -> Duplikate entfernen
print(f"Anzahl Duplikate: {df_train.duplicated().sum()}")
df_train = df_train.drop_duplicates()

# Entfernen von nicht relevanten Spalten
# TODO: Ist reservierungsstatus == storniert? Oder gibt es Abweichungen
# TODO: Bedeutung reservierungsstatus datum?  Wirklich irrelevant?
#           Beide Spalten fehlen im Testdatensatz und müssen daher entfernt werden
df_train = features(df_train, ["reservierungsstatus", "reservierungsstatus_datum"])

# Im Folgendem wird nur noch der Trainingsdatensatz betrachtet um Data Leakage zu vermeiden
X_train, X_valid, y_train, y_valid = train_test_split(
    df_train.drop(columns=["storniert"]),
    df_train["storniert"],
    test_size=0.2,
    random_state=42,
)

categorical = X_train.select_dtypes(exclude=np.number).columns
numerical = X_train.select_dtypes(include=np.number).columns

In [ ]:
# Ergebnis von Transformation der Länder prüfen
print(
    "Verteilung Kontinente (Trainingsdaten):\n"
    f"{X_train['herkunftskontinent'].value_counts()}\n\n"
    f"Anteil Gäste aus Portugal: "
    f"{X_train['aus_portugal'].astype(int).mean():.1%}\n"
)

# Ausgabe kategorischer Features
print(f"Kategorische Features ({len(categorical)}):\n")
for col in categorical:
    n_unique = X_train[col].nunique()
    print(f"{col:25} {n_unique:5} eindeutige Werte")

display("Trainingsdatensatz (kategorische Features):", X_train[categorical])

# Spalten die fehlende Werte enthalten ausgeben
missing = X_train.isna().sum()
display("Spalten mit fehlenden Werten:", missing[missing > 0])

# Ausreißern veranschaulichen
display(
    "Verteilung numerischer Features:",
    X_train.select_dtypes(include=np.number).describe(percentiles=[0.05, 0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.99, 0.995, 0.999]),
)

In [ ]:
import seaborn as sns
from sklearn.preprocessing import OneHotEncoder


def show_correlations(df: pd.DataFrame, target: pd.DataFrame, text: str) -> None:
    """Ausgabe der Korrelationen."""
    df_corr = df.copy()
    df_corr["storniert"] = target.copy()

    correlations = df_corr.corr()
    display(text, correlations["storniert"].sort_values(ascending=False))


show_correlations(X_train[numerical], y_train, "Korrelation der numerischen Features mit storniert:")

encoding = OneHotEncoder(handle_unknown="ignore", sparse_output=False)

# TODO: Entfernen nach Behandlung
categorical = categorical.drop("agentur_id")
df_encoded = encoding.fit_transform(X=X_train[categorical])
df_encoded = pd.DataFrame(df_encoded, columns=encoding.get_feature_names_out(), index=X_train.index)

show_correlations(df_encoded, y_train, "Korrelation der kategorischen Features mit storniert:")

# Korrelationsmatrix der Features
df_heatmap = pd.concat([X_train[numerical], df_encoded], axis=1)
df_heatmap["storniert"] = y_train

_, ax = plt.subplots(figsize=(20, 14))
sns.heatmap(
    df_heatmap.corr(),
    cmap="coolwarm",
    center=0,
    vmin=-1,
    vmax=1,
    linewidths=0.3,
    annot=False,
    ax=ax,
)
plt.title("Korrelationsmatrix", fontsize=14, pad=14)
plt.tight_layout()
plt.show()

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, StandardScaler


def clip_numericals(X: np.ndarray) -> np.ndarray:
    """Behandlung von Ausreißern."""
    lower = np.percentile(X, 1, axis=0)
    upper = np.percentile(X, 99, axis=0)

    return np.clip(X, lower, upper)


numerical_pipeline = Pipeline(
    [
        ("imputer", SimpleImputer(strategy="constant", fill_value=0.0)),  # Kinder (keine Angabe = keine Kinder)
        ("clip", FunctionTransformer(clip_numericals, validate=False)),
        ("scaler", StandardScaler()),  # Für logistische Regression
    ]
)

categorical_pipeline = Pipeline(
    [
        # ("imputer", SimpleImputer(strategy="constant", fill_value="Keine Angabe")),  # Herkunftsland und Agenturen
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ]
)

preprocessor = ColumnTransformer([("num", numerical_pipeline, numerical), ("cat", categorical_pipeline, categorical)])

# (4) Training der Modelle

In [ ]:
import gurobipy as gp
from gurobipy import GRB


def optimize_threshold_gurobi(y_true: pd.Series, y_prob: pd.Series, df_features: pd.DataFrame) -> tuple[float, float]:
    """Minimiert Gesamtkosten und berechnet Schwellwert."""
    aufenthaltsdauer = df_features["naechte_wochenende"] + df_features["naechte_wochentag"]
    fn_kosten = (df_features["durchschn_tagespreis"] * aufenthaltsdauer * 0.9).values

    y = y_true.values
    p = y_prob
    N = len(y)

    env = gp.Env(empty=True)
    env.setParam("OutputFlag", 0)
    env.start()
    m = gp.Model("Threshold_Optimierung", env=env)

    t = m.addVar(vtype=GRB.CONTINUOUS, lb=0.0, ub=1.0, name="threshold")
    z = m.addVars(N, vtype=GRB.BINARY, name="z")

    M = 1.01
    eps = 1e-5

    for i in range(N):
        m.addConstr(p[i] - t >= -M * (1 - z[i]), name=f"lower_{i}")
        m.addConstr(p[i] - t <= M * z[i] - eps, name=f"upper_{i}")

    kosten = gp.LinExpr()
    for i in range(N):
        if y[i] == 0:
            kosten += 250.0 * z[i]
        else:
            kosten += fn_kosten[i] * (1 - z[i])

    m.setObjective(kosten, GRB.MINIMIZE)
    m.optimize()

    if m.status == GRB.OPTIMAL:
        return m.ObjVal, t.X
    msg = "Gurobi konnte keine optimale Lösung finden."
    raise ValueError(msg)

In [ ]:
from sklearn.inspection import permutation_importance
from sklearn.metrics import accuracy_score, confusion_matrix, precision_score, recall_score
from sklearn.model_selection import GridSearchCV


def show_metrics(data: pd.DataFrame, pred: pd.DataFrame, title: str = "") -> None:
    """Ausgabe der relevanten Metriken."""
    tn, fp, fn, tp = confusion_matrix(data, pred).ravel()

    print(
        f"{title}\n"
        f"Accuracy:       {accuracy_score(data, pred):.7f}\n"
        f"Precision:      {precision_score(data, pred):.7f}\n"
        f"Recall:         {recall_score(data, pred):.7f}\n"
        f"False positive: {fp:.7f}\n"
        f"False negative: {fn:.7f}\n"
    )


def show_feature_importances(model, name: str) -> None:
    """Ausgabe der Feature Importances."""
    onehot = model.named_steps["preprocessor"].named_transformers_["cat"].named_steps["onehot"]
    feature_names_cat = onehot.get_feature_names_out(categorical)

    feature_names = numerical.tolist() + feature_names_cat.tolist()

    # Wichtigkeit der Features beim splitten
    importances = model.named_steps[name].feature_importances_

    df_importances = pd.DataFrame({"feature": feature_names, "importance": importances}).sort_values("importance", ascending=False)

    display("Feature Importances:", df_importances)


def show_permutation_importances(model) -> None:
    """Ausgabe der Permutation Importances."""
    result = permutation_importance(model, X_train, y_train, n_repeats=10, random_state=42, scoring="accuracy")

    df_importances = pd.DataFrame({"feature": X_train.columns, "importance": result.importances_mean}).sort_values(
        "importance", ascending=False
    )

    display("Permutation Importances:", df_importances)


def creative_function_name_here(name: str, ident: str, search: GridSearchCV, output_importances: bool = True) -> dict:
    """Ausgabe Modellergebnisse und Rückgabe Dictionary mit wichtigsten Parametern."""
    best_model = search.best_estimator_

    y_pred = best_model.predict(X_train)
    show_metrics(y_train, y_pred, title=f"{name} (Trainingsdaten)")

    y_pred = best_model.predict(X_valid)
    show_metrics(y_valid, y_pred, title=f"{name} (Validationsdaten)")

    display("Beste Parameter:", search.best_params_)

    if output_importances:
        show_feature_importances(best_model, ident)
    show_permutation_importances(best_model)

    # costs, threshold = optimize_threshold_gurobi(y_valid, y_pred, X_valid)
    costs, threshold = 0, 0

    return {
        name: {
            "Model": best_model,
            "Accuracy": round(accuracy_score(y_valid, y_pred, X_valid), 5),
            "Parameter": search.best_params_,
            "Gesamtkosten": costs,
            "Schwellwert": threshold,
        }
    }


models = []
cv = 2
verbose = 1

In [ ]:
from sklearn.tree import DecisionTreeClassifier

search = GridSearchCV(
    estimator=Pipeline([("preprocessor", preprocessor), ("tree", DecisionTreeClassifier(random_state=42))]),
    param_grid={
        "tree__max_depth": [15, 30],
        "tree__min_samples_split": [25, 100],
        "tree__min_samples_leaf": [20, 50],
        "tree__criterion": ["entropy"],
    },
    scoring="accuracy",
    cv=cv,
    n_jobs=-1,
    verbose=verbose,
)

search.fit(X_train, y_train)

models.append(creative_function_name_here("Entscheidungsbaum", "tree", search))

In [ ]:
from sklearn.linear_model import LogisticRegression

search = GridSearchCV(
    estimator=Pipeline([("preprocessor", preprocessor), ("logistic", LogisticRegression(random_state=42))]),
    param_grid={
        "logistic__C": [0.01, 0.1, 1, 10],
        "logistic__penalty": ["l2"],
        "logistic__solver": ["lbfgs"],
        "logistic__max_iter": [500],
    },
    scoring="accuracy",
    cv=cv,
    n_jobs=-1,
    verbose=verbose,
)

search.fit(X_train, y_train)

models.append(creative_function_name_here("Logistische Regression", "logistic", search, False))

In [ ]:
from sklearn.ensemble import RandomForestClassifier

search = GridSearchCV(
    estimator=Pipeline([("preprocessor", preprocessor), ("rf", RandomForestClassifier(random_state=42))]),
    param_grid={
        "rf__n_estimators": [100, 200],
        "rf__max_depth": [10, 20],
        "rf__min_samples_split": [2, 10],
        "rf__max_features": ["sqrt", "log2"],
    },
    scoring="accuracy",
    cv=cv,
    n_jobs=-1,
    verbose=verbose,
)

search.fit(X_train, y_train)

models.append(creative_function_name_here("Random Forest", "rf", search))

In [ ]:
from sklearn.ensemble import GradientBoostingClassifier

search = GridSearchCV(
    estimator=Pipeline([("preprocessor", preprocessor), ("gb", GradientBoostingClassifier(random_state=42))]),
    param_grid={
        "gb__n_estimators": [100],
        "gb__learning_rate": [0.05],
        "gb__max_depth": [3],
        "gb__min_samples_split": [2],
        "gb__min_samples_leaf": [1],
        "gb__subsample": [0.8, 1.0],
    },
    scoring="accuracy",
    cv=cv,
    n_jobs=-1,
    verbose=verbose,
)

search.fit(X_train, y_train)

models.append(creative_function_name_here("Gradient Boosting", "gb", search))

In [ ]:
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier, StackingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier

search = GridSearchCV(
    estimator=Pipeline(
        [
            ("preprocessor", preprocessor),
            (
                "stack",
                StackingClassifier(
                    estimators=[
                        ("tree", DecisionTreeClassifier(max_depth=10, min_samples_split=10, random_state=42)),
                        ("gb", GradientBoostingClassifier(n_estimators=150, learning_rate=0.1, max_depth=3, random_state=42)),
                        ("rf", RandomForestClassifier(n_estimators=100, max_depth=15, random_state=42)),
                    ],
                    final_estimator=LogisticRegression(max_iter=500, random_state=42),
                    cv=2,
                    n_jobs=-1,
                ),
            ),
        ]
    ),
    param_grid={"stack__final_estimator__C": [0.1, 1, 10], "stack__final_estimator__penalty": ["l2"]},
    scoring="accuracy",
    cv=cv,
    n_jobs=-1,
    verbose=verbose,
)

search.fit(X_train, y_train)

models.append(creative_function_name_here("Stacking", "stack", search, False))

In [ ]:
best_entry = max(models, key=lambda x: next(iter(x.values()))["Accuracy"])

best_model = next(iter(best_entry.values()))["Model"]

X_train["storniert"] = y_train
X_valid["storniert"] = y_valid

X_train["pred"] = None
X_valid["pred"] = None

for hotel in ["Stadthotel", "Ferienhotel"]:
    X_train_h = X_train[X_train["hotel_typ"] == hotel].copy()
    y_train_h = X_train_h["storniert"]
    X_train_feat = X_train_h.drop(columns=["storniert", "pred"])

    X_valid_h = X_valid[X_valid["hotel_typ"] == hotel].copy()
    y_valid_h = X_valid_h["storniert"]
    X_valid_feat = X_valid_h.drop(columns=["storniert", "pred"])

    best_model.fit(X_train_feat, y_train_h)

    y_pred = best_model.predict(X_train_feat)
    X_train.loc[X_train_h.index, "pred"] = y_pred

    y_pred = best_model.predict(X_valid_feat)
    X_valid.loc[X_valid_h.index, "pred"] = y_pred


show_metrics(y_train, X_train["pred"].astype(int), "Hotels getrennt (Trainingsdaten)")
show_metrics(y_valid, X_valid["pred"].astype(int), "Hotels getrennt (Validationsdaten)")

# costs, threshold = optimize_threshold_gurobi(y_valid, X_valid["pred"].astype(int), X_valid)
costs, threshold = 0, 0

models.append(
    {
        "Hotels getrennt": {
            "Model": best_model,
            "Accuracy": round(accuracy_score(y_valid, X_valid["pred"].astype(int)), 5),
            "Parameter": None,
            "Gesamtkosten": costs,
            "Schwellwert": threshold,
        }
    }
)

# (5) Evaluation

In [ ]:
lowest_cost_model = min(models, key=lambda x: next(iter(x.values()))["Gesamtkosten"])

rows = []

for model in models:
    name, values = next(iter(model.items()))

    rows.append(
        {
            "name": name,
            "Accuracy": values["Accuracy"],
            "Gesamtkosten": values["Gesamtkosten"],
            "Schwellwert": values["Schwellwert"],
        }
    )

df_models = pd.DataFrame(rows)

display("Modelle geordnet nach Gesamtkosten:", df_models.sort_values(by="Gesamtkosten"))

# (6) Vorhersage

In [ ]:
# Bestes Modell auf kompletten Datensatz trainieren
X = df_train.drop(columns=["storniert"])
y = df_train["storniert"]

best_entry = max(models, key=lambda x: next(iter(x.values()))["Accuracy"])

best_model = next(iter(best_entry.values()))["Model"]
best_model.fit(X, y)

In [ ]:
df_test = features(df_test)

y_pred = best_model.predict(df_test)

df_test["storniert_pred"] = y_pred
df_test[["buchungs_id", "storniert_pred"]].to_csv("data/vorhersage.csv", index=False)